In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 3.12 The Relativistic Formulation of Maxwell's Equations

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume III — Classical Electrodynamics",
    number="3.12",
    title="The Relativistic Formulation of Maxwell's Equations",
    blurb="The summit: spacetime, the field tensor, and Maxwell's four equations in "
    "two lines. Electricity and magnetism turn out to be a single object seen from "
    "different frames — magnetism is what an electric field looks like to a moving "
    "observer.",
    difficulty="advanced",
    estimate="180–220 min",
)

## Notebook overview

This is the longest and most demanding notebook of the volume, and the most rewarding.
It asks two unfamiliar things at once, to think in four dimensions and to think in
tensors, and in return the eleven notebooks before it snap into a single structure. The
claim is audacious and exact: **electricity and magnetism are not two phenomena but
one**, a single geometric object, and which part of it you call "electric" and which
"magnetic" depends only on how you are moving. Magnetism is what an electric field looks
like to a moving observer.

A note on reading order: this notebook sits in Volume III as the capstone of
electrodynamics, but it leans on special relativity. If you are meeting relativity for
the first time, you may prefer to read the special-relativity notebooks of Volume IV
([§4.1](../04-special-relativity/crisis-and-postulates.ipynb)–[§4.5](../04-special-relativity/four-momentum-energy.ipynb))
first and return here afterwards. We develop just enough relativity inline for
the notebook to stand on its own, but the fuller story is in Volume IV. In particular
the four-vector and `np.einsum` machinery we use inline here (the metric $\eta$, the
contraction $\eta_{\mu\nu}a^\mu b^\nu$, raising and lowering) is developed carefully, from
the geometry up, in [§4.3](../04-special-relativity/spacetime-minkowski.ipynb); a
reader who wants the index-string grammar spelled out
should look there.

The road there starts from the crisis [§3.8](maxwell-waves.ipynb) left open. Maxwell's
equations pick out a
speed $c=1/\sqrt{\mu_0\varepsilon_0}$, but a speed *relative to what*? Einstein's answer,
that $c$ is the same in every inertial frame and space and time themselves transform to
make it so, forces the Lorentz transformation, four-vectors, and a four-dimensional
spacetime. We develop only as much special relativity as we need to rewrite
electrodynamics; the full development is Volume IV, which we cross-reference. With that
groundwork we build the **field tensor** $F^{\mu\nu}$, the six numbers of $\mathbf E$
and $\mathbf B$ assembled into one antisymmetric object, collapse all four Maxwell
equations into **two tensor lines**, and watch a boost mix $\mathbf E$ and $\mathbf B$
into each other. We end with the two Lorentz invariants every observer agrees on, and
with the gauge arc's final turn: the freedom of [§3.6](magnetostatics.ipynb) and the
tool of [§3.8](maxwell-waves.ipynb) are revealed as
**structure**, $F^{\mu\nu}=\partial^\mu A^\nu-\partial^\nu A^\mu$ being manifestly
gauge-invariant, with a forward pointer to where gauge invariance becomes physics (Vol
V) and back to its Noether root ([§2.2](../02-classical-mechanics/noether.ipynb)).

We work in **SI units** with the metric signature $(-,+,+,+)$, used
throughout. (At this altitude Gaussian units are tidier, hiding the factors of $c$; we
keep SI for continuity and note where $c$ would vanish.) The one animation is genuinely
warranted: a boost continuously sweeping a pure electric field into a mixture of
electric and magnetic, the unification set in motion. Everything else is a still.

> **How to read the checks.** Each exercise ends with a `validate` call against an
> independent fact: the spacetime interval invariant under a boost, the four-velocity
> norm $-c^2$, the field tensor antisymmetric, the covariant equations reproducing the
> four 3-vector ones, the field invariants frame-independent, $F^{\mu\nu}$ unchanged by a
> gauge transformation. A ✓ is strong evidence; a ✗ is a prompt to *locate the
> discrepancy*, not a verdict.
>
> **Scope.** Enough relativity to recast electrodynamics, not a relativity course (that
> is Vol IV). See Nolting, *Theoretical Physics 3/4* {cite}`nolting3`; Griffiths,
> *Introduction to Electrodynamics* {cite}`griffiths_em` (ch. 12); Jackson
> {cite}`jackson` (ch. 11); Landau & Lifshitz, *The Classical Theory of Fields*
> {cite}`ll2`.

## Theory in brief

### The crisis that forces relativity

Maxwell's equations give one speed $c=1/\sqrt{\mu_0\varepsilon_0}$
([§3.8](maxwell-waves.ipynb)), but Galilean
velocity addition makes every speed frame-dependent. In which frame, then, is light's
speed $c$? The nineteenth-century answer, a luminiferous aether, failed (Michelson–
Morley). Einstein's resolution is the postulate

```{math}
:label: eq-crisis
c \text{ is the same in every inertial frame,}
```

from which space and time must themselves transform.

### Spacetime and the Lorentz transformation

Events live in four-dimensional spacetime, $x^\mu=(ct,x,y,z)$, with the invariant
interval set by the Minkowski metric $\eta=\mathrm{diag}(-1,1,1,1)$,

```{math}
:label: eq-lorentz
s^2 = \eta_{\mu\nu}x^\mu x^\nu = -c^2t^2+x^2+y^2+z^2 .
```

A boost along $x$ at speed $v$ ($\beta=v/c$, $\gamma=1/\sqrt{1-\beta^2}$) is the linear
map $\Lambda$ that mixes $t$ and $x$ while leaving $s^2$ unchanged.

### Four-vectors

Quantities that transform like $x^\mu$ are four-vectors: the four-velocity
$u^\mu=\gamma(c,\mathbf v)$ with the invariant norm $u^\mu u_\mu=-c^2$, and the two that
carry electrodynamics,

```{math}
:label: eq-four-vectors
J^\mu=(c\rho,\mathbf J), \qquad A^\mu=(V/c,\mathbf A).
```

Charge conservation and the Lorenz gauge each become a single four-divergence,
$\partial_\mu J^\mu=0$ and $\partial_\mu A^\mu=0$.

### The field tensor

The six numbers in $\mathbf E$ and $\mathbf B$ are not two 3-vectors but the six
independent components of one antisymmetric rank-2 tensor,

```{math}
:label: eq-field-tensor
F^{\mu\nu}=\partial^\mu A^\nu-\partial^\nu A^\mu, \qquad
F^{0i}=E_i/c, \quad F^{ij}=-\varepsilon^{ijk}B_k .
```

Written as a $4\times4$ matrix it holds $\mathbf E$ in its time row/column and $\mathbf
B$ in its spatial block: $\mathbf E$ and $\mathbf B$ are one object. Griffiths,
*Introduction to Electrodynamics*, ch. 12, works the identification of the components
out entry by entry.

### Maxwell in two lines

All four equations collapse to the sourced pair and the source-free (Bianchi) pair,

```{math}
:label: eq-covariant-maxwell
\partial_\mu F^{\mu\nu}=\mu_0 J^\nu, \qquad
\partial^\alpha F^{\beta\gamma}+\partial^\beta F^{\gamma\alpha}+\partial^\gamma F^{\alpha\beta}=0 .
```

The first reproduces Gauss ($\nu=0$) and Ampère–Maxwell ($\nu=i$); the second reproduces
$\nabla\cdot\mathbf B=0$ and Faraday.

### E and B mix under boosts

Transforming the tensor to a boosted frame, $F'^{\mu\nu}=\Lambda^\mu{}_\alpha
\Lambda^\nu{}_\beta F^{\alpha\beta}$, mixes the fields,

```{math}
:label: eq-EB-mixing
E'_\parallel=E_\parallel,\quad \mathbf E'_\perp=\gamma(\mathbf E+\mathbf v\times\mathbf B)_\perp,\quad
\mathbf B'_\perp=\gamma\Big(\mathbf B-\tfrac{1}{c^2}\mathbf v\times\mathbf E\Big)_\perp .
```

A pure Coulomb field, seen from a moving frame, acquires a magnetic component:
**magnetism is electricity in motion**.

### Gauge invariance as structure, and the invariants

Because $\partial^\mu\partial^\nu\chi$ is symmetric, $F^{\mu\nu}=\partial^\mu A^\nu-
\partial^\nu A^\mu$ is **manifestly unchanged** under

```{math}
:label: eq-gauge-structure
A^\mu \to A^\mu+\partial^\mu\chi ,
```

so the gauge freedom of [§3.6](magnetostatics.ipynb) and the gauge tool of
[§3.8](maxwell-waves.ipynb) are now seen as a structural
feature of the four-potential, with the Lorenz condition $\partial_\mu A^\mu=0$
manifestly Lorentz-invariant. From $F^{\mu\nu}$ one builds two Lorentz **scalars**,

```{math}
:label: eq-invariants
\mathbf E\cdot\mathbf B \;\propto\; F_{\mu\nu}\widetilde F^{\mu\nu}, \qquad
E^2-c^2B^2 \;\propto\; F_{\mu\nu}F^{\mu\nu},
```

the same in every frame even as $\mathbf E$ and $\mathbf B$ separately change. Landau &
Lifshitz, *The Classical Theory of Fields*, construct the dual tensor
$\widetilde F^{\mu\nu}$ and derive both invariants.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

from ecp import draw, validate
from ecp.animate import show

MU0 = 4.0e-7 * np.pi  # vacuum permeability, T·m/A
EPS0 = 8.8541878128e-12  # vacuum permittivity, F/m
C_LIGHT = 1.0 / np.sqrt(MU0 * EPS0)  # speed of light, m/s
ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
ETA = np.diag([-1.0, 1.0, 1.0, 1.0])  # Minkowski metric, signature (−,+,+,+)


def lorentz_boost(beta):
    """The 4×4 Lorentz boost matrix for a boost along x at β = v/c (eq-lorentz).

    Mixes ct and x while leaving y, z and the interval s^2 untouched; the
    relativistic replacement for a Galilean shear, with γ = 1/sqrt(1 − β^2).

    Parameters
    ----------
    beta : float
        Boost speed as a fraction of c (|β| < 1).

    Returns
    -------
    numpy.ndarray
        The 4×4 boost Λ^μ_ν acting on x^μ = (ct, x, y, z).
    """
    g = 1.0 / np.sqrt(1.0 - beta**2)
    L = np.eye(4)
    L[0, 0] = g
    L[0, 1] = L[1, 0] = -g * beta
    L[1, 1] = g
    return L


def field_tensor(E, B):
    """The contravariant electromagnetic field tensor F^μν from E and B (eq-field-tensor).

    Packs the six field components into one antisymmetric 4×4 matrix: E in the
    time row/column (F^{0i} = E_i/c) and B in the spatial block
    (F^{ij} = −ε^{ijk} B_k). The single object that E and B are two faces of.

    Parameters
    ----------
    E : array_like
        Electric field (Ex, Ey, Ez), in V/m.
    B : array_like
        Magnetic field (Bx, By, Bz), in T.

    Returns
    -------
    numpy.ndarray
        The 4×4 tensor F^μν.
    """
    Ex, Ey, Ez = E
    Bx, By, Bz = B
    c = C_LIGHT
    return np.array(
        [
            [0.0, Ex / c, Ey / c, Ez / c],
            [-Ex / c, 0.0, Bz, -By],
            [-Ey / c, -Bz, 0.0, Bx],
            [-Ez / c, By, -Bx, 0.0],
        ]
    )


def extract_EB(F):
    """Read the electric and magnetic fields back out of a field tensor F^μν.

    The inverse of :func:`field_tensor`: E_i = c·F^{0i} and
    B_k = −(1/2)·ε_{kij}·F^{ij}.

    Parameters
    ----------
    F : numpy.ndarray
        A 4×4 field tensor.

    Returns
    -------
    E, B : numpy.ndarray
        The electric field (V/m) and magnetic field (T).
    """
    c = C_LIGHT
    E = c * np.array([F[0, 1], F[0, 2], F[0, 3]])
    B = np.array([F[2, 3], F[3, 1], F[1, 2]])  # Bx=F^{23}, By=F^{31}, Bz=F^{12}
    return E, B


def four_velocity(v):
    """The four-velocity u^μ = γ(c, v) (eq-four-vectors).

    The tangent to a worldline; its Minkowski norm is the invariant −c^2, the
    relativistic statement that everything moves through spacetime at speed c.

    Parameters
    ----------
    v : array_like
        Ordinary 3-velocity (vx, vy, vz), in m/s.

    Returns
    -------
    numpy.ndarray
        The four-velocity (γc, γv).
    """
    v = np.asarray(v, float)
    g = 1.0 / np.sqrt(1.0 - (v @ v) / C_LIGHT**2)
    return np.concatenate([[g * C_LIGHT], g * v])


def minkowski_dot(a, b):
    """Minkowski inner product a^μ η_μν b^ν in signature (−,+,+,+).

    Parameters
    ----------
    a, b : array_like
        Four-vectors.

    Returns
    -------
    float
        The invariant scalar product (``np.einsum`` over the metric).
    """
    return float(np.einsum("m,mn,n->", a, ETA, b))

## Movement I — Spacetime and four-vectors

## Exercise 1 — The Lorentz transformation and the invariant interval

Special relativity begins by taking $c$ as absolute {eq}`eq-crisis` and letting space
and time bend to keep it so. The carrier of that bending is the **Lorentz boost**
$\Lambda$, which mixes $t$ and $x$ but preserves the interval $s^2=\eta_{\mu\nu}x^\mu
x^\nu$ {eq}`eq-lorentz` ({numref}`fig-rm-spacetime`). What looks like a fixed time and a
fixed position to one observer is a blend of both to another, yet both agree on $s^2$.

**This exercise (worked).**

1. Construct the $4\times4$ boost matrix $\Lambda(v)$ at $v=0.6c$ as an
   explicit `numpy` array (entries $\gamma$ and $-\gamma\beta$ on the
   $(0,0),(0,1),(1,0),(1,1)$ slots), apply it to a set of events by matrix
   multiplication `events @ Λ.T`, and form the interval as the contraction
   `x @ η @ x` to confirm it is unchanged while $t$ and $x$ individually
   transform.
2. Check that a light ray (an event on the line $x=ct$) maps to another
   light ray: the speed of light is $c$ in both frames, the whole point.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    s2_after, s2_before, "the spacetime interval is Lorentz-invariant", rtol=1e-10
)
validate.close(
    light_after[0],
    light_after[1],
    "a light ray maps to a light ray — c is the same in both frames",
    rtol=1e-10,
)

## Exercise 2 — Four-vectors: velocity, current, potential

Anything that transforms under $\Lambda$ like the position $x^\mu$ is a **four-vector**,
and the Minkowski norm $a^\mu a_\mu$ is then a scalar every observer agrees on. The
four-velocity $u^\mu=\gamma(c,\mathbf v)$ {eq}`eq-four-vectors` has the fixed norm
$u^\mu u_\mu=-c^2$, a compact way of saying everything advances through spacetime at
speed $c$. The same packaging gives the **four-current** $J^\mu=(c\rho,\mathbf J)$ and
**four-potential** $A^\mu=(V/c,\mathbf A)$, and in this language two scattered facts of
[§3.8](maxwell-waves.ipynb), charge conservation and the Lorenz gauge, each become a
single four-divergence.

**This exercise (worked).**

1. Construct $u^\mu$ for $v=0.6c$ as a `numpy` 4-array and verify
   $u^\mu u_\mu=-c^2$ by contracting it with the metric,
   `np.einsum("m,mn,n->", u, η, u)`.
2. Take a plane-wave four-potential and form the four-divergence
   $\partial_\mu A^\mu$ by differencing each component with
   `numpy.gradient` along its axis and summing, confirming the Lorenz
   condition holds — the single equation $\partial_\mu A^\mu=0$ standing in
   for $\nabla\cdot\mathbf A+\tfrac1{c^2}\partial_t V=0$.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    u_norm, -(C_LIGHT**2), "the four-velocity has invariant norm −c²", rtol=1e-10
)
validate.close(
    lorenz_residual,
    0.0,
    "the Lorenz gauge ∂_μA^μ = 0 is a single four-divergence",
    atol=1e-6,
)

## Movement II — The field tensor and Maxwell

## Exercise 3 — The field tensor $F^{\mu\nu}$

Here is the centerpiece. The six numbers we have carried as two separate 3-vectors,
$\mathbf E$ and $\mathbf B$, are really the six independent entries of one antisymmetric
$4\times4$ tensor $F^{\mu\nu}$ {eq}`eq-field-tensor`, with $\mathbf E$ along the time
row and column and $\mathbf B$ in the spatial block ({numref}`fig-rm-tensor`). The
antisymmetry $F^{\mu\nu}=-F^{\nu\mu}$ leaves exactly six free components, precisely the
count of $\mathbf E$ and $\mathbf B$ together. They were never two things.

**This exercise (worked).** Construct $F^{\mu\nu}$ from a stated $\mathbf E$ and
$\mathbf B$ as an explicit $4\times4$ `numpy` array (with $E_i/c$ in the time row/column
and $-\varepsilon^{ijk}B_k$ in the spatial block), confirm antisymmetry with
`np.allclose(F, -F.T)` (here read off as $\max|F+F^{\mathsf T}|$), and extract $\mathbf
E$ and $\mathbf B$ back out, a clean round trip. Six components, one object.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    antisym_residual,
    0.0,
    "the field tensor is antisymmetric — E and B are its six components",
    atol=1e-12,
)
validate.close(
    np.concatenate([E_back, B_back]),
    np.concatenate([E_test, B_test]),
    "E and B round-trip through F^μν exactly",
    rtol=1e-12,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — Maxwell's equations in covariant form

Now the collapse. The two tensor equations {eq}`eq-covariant-maxwell` contain all four
of Maxwell's laws. The sourced equation $\partial_\mu F^{\mu\nu}=\mu_0 J^\nu$ gives
Gauss's law for $\nu=0$ and Ampère–Maxwell for $\nu=i$; the Bianchi identity (the
source-free pair) gives $\nabla\cdot\mathbf B=0$ and Faraday. Eleven notebooks of
separate laws, written in two lines.

**This exercise (worked).** Verify the unpacking numerically. On a smooth test field
$\mathbf E(t,\mathbf r)$, $\mathbf B(t,\mathbf r)$, assemble $F^{\mu\nu}$ component-wise
on a 4-D grid, then form the four-divergence $\partial_\mu F^{\mu\nu}$ by differentiating
each $F^{\mu\nu}$ along its $x^\mu$ axis with `numpy.gradient` (with $\partial_0=\partial
/\partial(ct)$) and summing over $\mu$, and confirm component-by-component that it equals the
corresponding 3-vector operator: $\partial_\mu F^{\mu0}=-(\nabla\cdot\mathbf E)/c$
(Gauss) and $\partial_\mu F^{\mu i}=(\nabla\times\mathbf B-\tfrac1{c^2}\partial_t\mathbf
E)_i$ (Ampère–Maxwell), each to finite-difference precision on the interior. The tensor
equation *is* the 3-vector equations.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    divF_0[core],
    gauss_3v[core],
    "∂_μF^μ0 = μ₀J⁰ reproduces Gauss's law (∇·E component-wise)",
    atol=1e-9,
)
validate.close(
    divF_1[core],
    ampere_x_3v[core],
    "∂_μF^μi = μ₀J^i reproduces Ampère–Maxwell (component-wise)",
    atol=1e-9,
)

## Movement III — The unification: E and B mix

## Exercise 5 — Boosting the field tensor

Because $F^{\mu\nu}$ is a tensor, a change of frame acts on it by the boost on each
index, $F'^{\mu\nu}=\Lambda^\mu{}_\alpha\Lambda^\nu{}_\beta F^{\alpha\beta}$, i.e.
$F'=\Lambda F\Lambda^{\mathsf T}$ {eq}`eq-EB-mixing`. Reading $\mathbf E'$ and $\mathbf
B'$ out of the boosted tensor must reproduce the textbook transformation rules, with the
parallel components unchanged and the perpendicular ones mixing $\mathbf E$ and $\mathbf
B$.

**This exercise (worked).** Boost $F^{\mu\nu}$ to a frame at $v=0.6c$ by contracting both
indices with the boost, `np.einsum('ma,nb,ab->mn', Λ, Λ, F)` (equivalently $\Lambda F
\Lambda^{\mathsf T}$), extract $\mathbf E'$ and $\mathbf B'$, and check them against the closed-form rules
$E'_y=\gamma(E_y-vB_z)$, $B'_z=\gamma(B_z-vE_y/c^2)$, and their partners. Tensor algebra
and the hand-derived formulas must agree to machine precision.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    Ep_tensor,
    Ep_formula,
    "the boosted field tensor reproduces the E transformation law",
    rtol=1e-8,
)
validate.close(
    Bp_tensor,
    Bp_formula,
    "the boosted field tensor reproduces the B transformation law",
    rtol=1e-8,
)

## Exercise 6 — Magnetism is electricity in another frame

This is the volume's deepest result. Take a **pure electric field**, $\mathbf B=0$, the
Coulomb field of a static charge, and look at it from a moving frame. A genuine
**magnetic** field appears: $\mathbf B'\neq0$ {eq}`eq-EB-mixing`. There is no separate
"magnetism" waiting in the wings; the magnetic field of a moving charge
([§3.6](magnetostatics.ipynb)) simply
*is* its electric field, seen by an observer in motion ({numref}`fig-rm-boost-anim`).

**This exercise (worked).**

1. Construct the pure-$E$ field tensor from $\mathbf E=(0,10^5,0)\,$V/m and
   $\mathbf B=0$, apply the perpendicular boost at $v=0.6c$ with the same
   `np.einsum('ma,nb,ab->mn', Λ, Λ, F)` contraction, and read $\mathbf B'$
   back from the transformed tensor.
2. Show the magnetic field that emerges equals the closed form
   $B'_z=-\gamma vE_y/c^2$.

The animation ({numref}`fig-rm-boost-anim`) sweeps the boost from $0$ to
$0.9c$ and watches $\mathbf B'$ grow from nothing as the frame speeds up,
the unification set in motion.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    Bp_pure[2],
    -g * v * E_pure[1] / C_LIGHT**2,
    "a pure electric field acquires exactly B'_z = −γvE_y/c² under a boost — "
    "magnetism is relativity",
    rtol=1e-8,
)
validate.close(
    np.array([Bp_pure[0], Bp_pure[1]]),
    np.zeros(2),
    "the boost generates no magnetic component along x or y (geometry of v × E)",
    atol=1e-15,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — The current-carrying wire, demystified

The most famous demonstration that magnetism is relativity is a neutral wire. In the lab
it carries a current and exerts a **magnetic** force on a nearby moving charge. But in
the charge's own rest frame there is no motion of the charge to feel a magnetic force;
instead, the wire's positive and negative charge densities Lorentz-contract by different
amounts, so the wire appears **charged**, and the same force arrives as an **electric**
one. Two descriptions, one physics.

**This exercise (student).** Model the wire as positive ions at rest and electrons
drifting at $v_d$, neutral in the lab. With plain `numpy` scalar arithmetic on the
stated values, compute the lab-frame magnetic force $F=quB$ with $B=\mu_0 I/2\pi r$
([§3.6](magnetostatics.ipynb)); then transform the electron velocity to the charge's
frame with the relativistic
velocity-addition formula, Lorentz-contract each charge density by its own $\gamma$, sum
them into a net density $\lambda_{\rm net}$, and compute the rest-frame electric force
$F'=qE'$ with $E'=\lambda_{\rm net}/2\pi\varepsilon_0 r$; confirm the two agree after the
transverse-force factor $\gamma_u$.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    F_elec_rest / gu,
    F_mag_lab,
    "the lab magnetic force equals the electric force in the charge's rest frame",
    rtol=1e-3,
)

## Movement IV — Invariants and structure

## Exercise 8 — The Lorentz invariants

Although $\mathbf E$ and $\mathbf B$ each change from frame to frame, two combinations
of them do not {eq}`eq-invariants`: the scalars $\mathbf E\cdot\mathbf B$ and
$E^2-c^2B^2$, built from $F_{\mu\nu}\widetilde F^{\mu\nu}$ and $F_{\mu\nu}F^{\mu\nu}$,
are the same in every frame. Whether a field is "more electric" or "more magnetic" is a
matter of who is looking, but these two numbers are absolute, and they classify the
field: a radiation field, for instance, has both invariants zero for every observer.

**This exercise (worked).** Form $\mathbf E\cdot\mathbf B$ as the dot product
`np.dot(E, B)` and $E^2-c^2B^2$ as `E @ E - c**2 * (B @ B)` in the lab frame and in the
boosted frame of Exercise 5, and confirm both are unchanged even though $\mathbf E$ and
$\mathbf B$ themselves are not.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    np.array([EdotB_boost, inv2_boost]),
    np.array([EdotB_lab, inv2_lab]),
    "the two field invariants E·B and E²−c²B² are the same in all frames",
    rtol=1e-6,
)

## Exercise 9 — Gauge invariance as structure

The gauge arc reaches its summit. Because $F^{\mu\nu}=\partial^\mu A^\nu-\partial^\nu
A^\mu$ and partial derivatives commute, adding the gradient of any scalar to the
four-potential, $A^\mu\to A^\mu+\partial^\mu\chi$ {eq}`eq-gauge-structure`, changes
nothing: $\partial^\mu\partial^\nu\chi-\partial^\nu\partial^\mu\chi=0$. The gauge freedom
that was a *freedom* in [§3.6](magnetostatics.ipynb) and a *tool* in
[§3.8](maxwell-waves.ipynb) is now seen as built into the very
definition of the field, **structure**. And the Lorenz condition $\partial_\mu A^\mu=0$,
being a four-divergence, is manifestly Lorentz-invariant, where the Coulomb gauge
$\nabla\cdot\mathbf A=0$ is not.

**This exercise (worked).** On a grid, build $F^{\mu\nu}$ from a four-potential by
differencing it with `numpy.gradient` (raising indices with the metric), add
$\partial^\mu\chi$ for a stated scalar $\chi$ (again via `numpy.gradient`), rebuild
$F^{\mu\nu}$ the same way, and confirm the two are identical with `np.allclose` (here the
max difference on the interior). The arc closes: **freedom
([§3.6](magnetostatics.ipynb)) → tool ([§3.8](maxwell-waves.ipynb)) → structure
(here) → physics (Vol VI, Aharonov–Bohm)**, rooted in the symmetry–conservation link of
Noether ([§2.2](../02-classical-mechanics/noether.ipynb)), where a global phase
symmetry gives charge conservation and its local
version is gauge symmetry itself.

In [ ]:
# (solution hidden on the public site)


### Validation 9

In [ ]:
validate.close(
    F_after[core2],
    F_before[core2],
    "F^μν is invariant under a gauge transformation A → A + ∂χ (gauge as structure)",
    atol=1e-9,
)

## Exercise 10 — What the volume built

Stand at the summit and look back. The volume opened with a single static charge and the
question of what it does to the space around it ([§3.1](coulomb-field.ipynb)). It
found a field, then a potential, then a local law (Gauss, [§3.3](gauss-law.ipynb)); it
added magnetism ([§3.6](magnetostatics.ipynb)), coupled the two
through induction ([§3.7](induction.ipynb)), and closed the set with the displacement
current into Maxwell's
equations and the discovery that light is an electromagnetic wave
([§3.8](maxwell-waves.ipynb)). It confined
those waves ([§3.9](waveguides-cavities.ipynb)), found what produces them
([§3.10](radiation.ipynb)), and met them again as circuits
([§3.11](rlc-circuits.ipynb)). And here, at the end, all of it, eleven notebooks of
separate laws, collapses
into two tensor equations for a single object $F^{\mu\nu}$ in a four-dimensional
spacetime. Electrodynamics is revealed as one **relativistic field theory**, the first
and the exemplar of classical field theory.

The crisis that drove us, "$c$ relative to what?", is resolved: there is no preferred
frame, $\mathbf E$ and $\mathbf B$ are one object, and the laws read the same for every
observer. The road continues to special relativity in full (Vol IV), to quantum
electrodynamics where gauge invariance becomes physics (Vol VI), and to field theory
beyond. The volume began by asking what a charge does to the space around it, and ends
by finding that space, time, electricity, and magnetism are one structure.

**This exercise.** Confirm the summit numerically in one line: the same field
configuration, viewed in the lab and in a boosted frame, has different $\mathbf E$ and
$\mathbf B$ but identical invariants and an identical, antisymmetric $F^{\mu\nu}$
structure, the single object underneath the two appearances.

In [ ]:
# (solution hidden on the public site)


### Validation 10

In [ ]:
validate.check(
    one_field,
    "one electromagnetic field: frame-independent invariants and an antisymmetric F^μν",
)

## Notebook summary

- **Spacetime and four-vectors.** The boost $\Lambda(0.6c)$ preserves the interval
  $s^2=\eta_{\mu\nu}x^\mu x^\nu$ and maps light rays to light rays ($c$ invariant); the
  four-velocity has norm $u^\mu u_\mu=-c^2$, and the Lorenz gauge is the single
  four-divergence $\partial_\mu A^\mu=0$.
- **The field tensor.** $\mathbf E$ and $\mathbf B$ are the six components of one
  antisymmetric $F^{\mu\nu}$ {eq}`eq-field-tensor` (round-tripped exactly), and Maxwell's
  four equations are the two tensor lines $\partial_\mu F^{\mu\nu}=\mu_0 J^\nu$ and the
  Bianchi identity, verified component-by-component against $\nabla\cdot\mathbf E$ and
  $\nabla\times\mathbf B-\tfrac1{c^2}\partial_t\mathbf E$.
- **The unification.** Boosting $F^{\mu\nu}$ reproduces the $\mathbf E,\mathbf B$
  transformation laws to machine precision; a **pure electric field acquires a magnetic
  one** under a boost ($B'_z=-\gamma vE_y/c^2$, animated), and the current-carrying wire's
  magnetic force is an electric force in the charge's frame. Magnetism is electricity in
  motion.
- **Invariants and structure.** $\mathbf E\cdot\mathbf B$ and $E^2-c^2B^2$ are the same
  in every frame; $F^{\mu\nu}=\partial^\mu A^\nu-\partial^\nu A^\mu$ is manifestly
  gauge-invariant, closing the arc **freedom ([§3.6](magnetostatics.ipynb)) → tool
  ([§3.8](maxwell-waves.ipynb)) → structure (here) →
  physics (Vol VI)**, rooted in Noether
  ([§2.2](../02-classical-mechanics/noether.ipynb)). From a static charge
  ([§3.1](coulomb-field.ipynb)) to one
  relativistic field theory: **Volume III complete**.

## Outlook

- **Special relativity in full (Vol IV).** Kinematics, dynamics, $E=mc^2$, and the
  spacetime geometry developed here only as far as electrodynamics required.
- **The stress–energy tensor.** $F^{\mu\nu}$ also builds $T^{\mu\nu}$, the energy,
  momentum, and stress carried by the field itself, the source of gravity in general
  relativity.
- **Relativistic radiation.** The Liénard–Wiechert potentials extend the radiation of
  [§3.10](radiation.ipynb) to fast charges, giving synchrotron light and the
  relativistic beaming of
  accelerators.
- **Gauge theory and the Standard Model.** Promoting the global phase symmetry of
  Noether ([§2.2](../02-classical-mechanics/noether.ipynb)) to a local one *is*
  electromagnetism; generalising the group leads to
  Yang–Mills theory and the Standard Model, far beyond this course's arc.
- **Quantum electrodynamics (Vol VI).** Gauge invariance becomes physical in the
  Aharonov–Bohm effect, and minimal coupling weds the four-potential to the quantum
  particle, where this volume's structure becomes the language of modern physics.

In [ ]:
from ecp.style import footer

footer()